# Report

## Data
The `train_users` was the primary data file for initial exploration.  A basic review of the data was performed (just "look" at the data).  It was apparent that `country_destination` was the target variable.  Columns found to be text were inspected for unique values and value counts.  One of the key items here is if a variable has a low count, when spliting the data (more on that in modeling), the desire is to have enough samples with that category present.  Basically any cateogry occuring less than 20 times was grouped into an "other" category for that feature.

Several features were missing data.  The target had an unusual value of "NDF" as well.  Missing features were imputed with the median value.  An examination of the `Age` category showed the mean and median to be quite close--however further inspection did seem to show that the data was not "clean" or that there were errors in the Age category.  Some were obvious such as `1957` (Best guess is someone but in their birth year, not their age).  Looking at the distribution of age, anything outside of 18-90 was determined to be missing or inaccurate (how many people under 18 are booking travel? and based on age distribution, how many over 90!)

Going back to the target, it was clarifed that "NDF", as initially suspected, represented a missing value (No Destination Found).  For modeling purposes, these rows were excluded.  The reason being if the desire is to predict the first selected destination, and those rows have not chosen, they do not fall within the criteria for the model (They may however, make for an excellent continuing test set as they do select their first destination!)

Finally there were several date and timestamp columns.  All of these were converted to Unix Time (Elapsed since Jan 1, 1970) and treated as continuous features, even though initially they were text.

All the other Features were one hot encoded with all instances less than 20 grouped together in an "other" category.  The target was encoded as an integer to repesent a multiclass classifier. However, a reminder for the target Class "2" is not twice Class "1", for the target the integer is shorthand for a vector that repesents the classes independantly.  When finished there was 75 columns total in the dataset including the target and user_id.  The user_id was not used as a feature (it was dropped) and `country_destination` was used as the target, resulting in 73 input features.  There were 88908 samples total used to model

## Modeling
Given the quick nature of the project and the need to build an API, a Random Forest was selected as an initial model. The reason for this are it is a powerful non-linear model, that does not require feature scaling, easy to avoid overfitting, fairly speedy, and sets a good target to beat for other models like XGBoost, SVM, and Neural Networks.

A Baseline run of the Random Forest showed a very heavily biased dataset with sever overfitting.  Because the dataset is dominated by peoplpe choosing the US as a destination, the model initial heavily favored predicting "US" for almost every client.  Confusion Matrix results and classification report showed strong bias towards a US prediction and fairly weak ability to predict any other class.

The first task was to tune to Random Forest for improvement.  Class Weight was set to balanced to help deal with the heavily biased "US" target (aka other classes counted "more" when building the model)  Hyperparameter tuning was done by selecting 20 combinations at random from the following table of hyperparameters.  Evaluation was done with a stratified K Fold split with 5 folds.  Thus for every combination of parameters 5 models were built, each model using 80% of the data to train and predicting the remaining 20%.  But using all the folds iteratively it is possible to estimate performance on the entire dataset by combing when each fold was used as a hold out test set and combining of of the out of fold data into one prediction set.

| Parameter | Values | 
| -------- | -------- |
| max_depth | 5, 10, 15, None (unlimited) | 
| min_samples_split | 4, 6, 8, 10 |
| min_samples_leaf | 1, 2, 4 |
| max_features | 5, 10, 15, 20, 'sqrt' (8) |

`Balanced_Accuracy` was the scoring metric used (maximum score).  Rather than standard accuracy, due to the heavily biased US results, balanced accuracy should assist in predicting the minority classes. (`F1_Macro` is also a choice for this).  The best performing model has bot max_depth and max_feautres of 10 and minimum samples as 4 for both split and leaf.  The balanced accuracy was just under 13% (12.8).  Not stellar at all, but a starting point.

Given this model, feature importance was examined.  The timestamp features show to be very influential, suggesting that they may be a target for feature creation, if time would allow.

In order to improve the results XGBoost modeling was attempted.  The results saw very little improvement.  Although trained on multi-logloss and trying early stopping for both loss and various metrics (error, f1, top-k accuracy), XGBoost showed very little promise, often terminating via early stopping well before 100 rounds, which experience has shown to be a sign that XGBoost was not performning well.  Combined with the shortcomings of Random Forest, rather than continue with more advanced models, the next step is probably to attempt feature creation with the timestamps and additional user session data.

The classification report and confusion matrix below show the strong bias of the US class.  While technically the 'US' had many false positives, this was due to the class weights which prioritized the "off classes".  When class weights were disabled, the opposite became true, all the predicitions were US.  This is the challange of this dataset, find the balance between the under represented classes and the heavy US destination population.

![Confusion Matrix](CM.png)
![Classification Report](class\ report.png)

## API

With a model now able to produce predictions (albeit, not terribly impressive), the decision was made to use FastAPI to create an API for the model.  The Random Forest was trained on best parameters using the entire dataset.  The model was exported/saved using SKOPS once it was trained.  The saved model was loaded into FastAPI, as was the entire userdataset.  With only hundreds of thousands of entries, using memory for storage is low risk.  However if the entried began to scale, rather than loading data into memory, a database should be used.  Using the FastAPI template a model prediction is made by first retreiving the data stored in memory (as a dataframe), running data through the model to get an integer prediction, that integer used as a lookup for the country represented and the country was returned as a string.

a sample curl request verified functionality
```
curl -X POST "http://127.0.0.1:8000/predict"      
     -H "Content-Type: application/json"
     -d '{"username": "4ft3gnwmtx"}'
```
This sample returned 'other' or class 2, the same class predicted when modeling verifying that the api is working.

## Running the API
To start the api. the files may be pulled from `https://github.com/rdslater/ctgproject`.  The data files should be placed in a subfolder /data with without changing the name.  To setup the API to run the following command is issued `uvicorn fastapidemo:app`.  Using the above curl request will complete the functionality

## Considerations
No time was given to the user session to explore feature creation.  Given the need to do EDA/Modeling/Deployment only very basic effort was made to try different models and feature creation.  It is expected that feature creation will give the biggest improvement in performance. XGBoost is one of the top performing algorithms, so it is felt that there is no need to explore an SVM or Neural Network.  Tuning of XGBoost was not much explored.  Given the already low performance, time would be better spent creating features.  As mentioned previous, the timestamp columns showed very high feature importance--along with user sessions, the is likely where the largest improvement would been seen in terms of time/effort.  A top-k accuracy with k=3 did show promise with over 40% accuracy (likely target was in the top 3 40% of the time).  It would be helpful to understand if the end user of this system would be amenable to such a style of prediction: rather than a single destination, here are the top 3 choices (3 being somewhat arbitrary).
